# SECOM Yield Prediction — XGBoost Model

**Date:** September 7, 2026  
**Depends on:** 02_preprocessing.ipynb, 04_random_forest.ipynb, 05_shap_interpretability.ipynb

## Motivation

Random Forest achieved PR-AUC = 0.2743 on the test set with a severe
overfitting gap (train PR-AUC = 1.000). The model correctly caught 13 of
21 failures at threshold 0.20 but missed 8 failures. SHAP analysis
identified a masking effect in at least one missed failure (Wafer with
F31/F59/F103) where a within-spec sensor suppressed genuine failure signals.

## Hypothesis

XGBoost's sequential error correction will better handle cases where
competing sensor signals obscure failure patterns. Explicit L1/L2
regularization (alpha, lambda) will reduce the overfitting gap observed
in Random Forest. Combined, these properties should improve both
PR-AUC and recall on missed failures.

## What XGBoost does differently

- Boosting vs. bagging: trees built sequentially, each correcting
  previous errors (vs. RF's parallel independent trees)
- Explicit regularization: lambda (L2), alpha (L1), and max_depth
  directly control model complexity
- Scale-invariant: handles the scaled feature space effectively
- Native imbalance handling: scale_pos_weight parameter


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import joblib
import shap
import warnings
warnings.filterwarnings('ignore')

import xgboost as xgb
from sklearn.metrics import (
    classification_report, confusion_matrix,
    average_precision_score, roc_auc_score,
    precision_recall_curve, roc_curve
)

PROCESSED = Path('../data/processed')
FIGURES   = Path('../reports/figures')

X_train = pd.read_csv(PROCESSED / 'X_train.csv')
X_test  = pd.read_csv(PROCESSED / 'X_test.csv')
y_train = pd.read_csv(PROCESSED / 'y_train.csv').squeeze()
y_test  = pd.read_csv(PROCESSED / 'y_test.csv').squeeze()

scaler     = joblib.load(PROCESSED / 'scaler.pkl')
X_train_sc = pd.DataFrame(scaler.transform(X_train), columns=X_train.columns)
X_test_sc  = pd.DataFrame(scaler.transform(X_test),  columns=X_test.columns)

# Class imbalance ratio for scale_pos_weight
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale_pos_weight = neg / pos

print(f"Negative (pass) samples: {neg}")
print(f"Positive (fail) samples: {pos}")
print(f"scale_pos_weight: {scale_pos_weight:.2f}")
print(f"Interpretation: each failure weighted as {scale_pos_weight:.1f} passes")

Negative (pass) samples: 1170
Positive (fail) samples: 83
scale_pos_weight: 14.10
Interpretation: each failure weighted as 14.1 passes


## XGBoost Hyperparameter Decisions

**n_estimators = 500**  
More trees than RF because boosting uses shallow trees — each
contributes less individually, requiring more of them.

**max_depth = 4**  
Shallow trees (vs. RF's unlimited depth). This is the primary
regularization mechanism. Depth 4 allows 2-way and 3-way feature
interactions while preventing the memorization seen in RF.

**learning_rate = 0.05**  
Also called eta. Small learning rate means each tree makes a
conservative correction — slower learning but better generalization.
Standard range: 0.01–0.3. We choose 0.05 as a moderate value.

**subsample = 0.8**  
Each tree is trained on 80% of randomly sampled training data.
Introduces randomness similar to RF — reduces overfitting.

**colsample_bytree = 0.8**  
Each tree uses 80% of randomly sampled features.
Similar to RF's max_features — decorrelates trees.

**scale_pos_weight = neg/pos ≈ 15**  
XGBoost's native class imbalance handler. Equivalent to
class_weight='balanced' in sklearn. Mathematically: the gradient
for positive (failure) samples is multiplied by this weight.

**reg_alpha = 0.1 (L1 regularization)**  
Encourages sparse feature weights — some features will have
exactly zero contribution. Appropriate given our 446-feature
high-dimensional space.

**reg_lambda = 1.0 (L2 regularization)**  
Penalizes large weights — prevents any single feature from
dominating. Default value; we retain it deliberately.

**eval_metric = 'aucpr'**  
Optimize for PR-AUC during training — consistent with our
primary evaluation metric. XGBoost will report this on the
validation set at each boosting round if we use early stopping.


## Early Stopping

We use early stopping with a validation set to prevent overfitting
during boosting. After each tree is added, XGBoost evaluates
performance on a held-out validation set (20% of training data).
If PR-AUC does not improve for 50 consecutive rounds, training stops.

This means n_estimators=500 is a maximum, not a fixed number.
The actual number of trees used is determined by the data.
We report the actual number of trees used in our results.


In [3]:
from sklearn.model_selection import train_test_split

# Create internal validation set from training data for early stopping
# This does NOT touch our held-out test set
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_sc, y_train,
    test_size=0.15,
    random_state=42,
    stratify=y_train
)

print(f"Training (for XGB):   {X_tr.shape} | Failures: {y_tr.sum()}")
print(f"Validation (early stop): {X_val.shape} | Failures: {y_val.sum()}")
print(f"Test (held-out):      {X_test_sc.shape} | Failures: {y_test.sum()}")

xgb_model = xgb.XGBClassifier(
    n_estimators=500,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    reg_alpha=0.1,
    reg_lambda=1.0,
    eval_metric='aucpr',
    early_stopping_rounds=50,
    random_state=42,
    n_jobs=-1,
    verbosity=0
)

xgb_model.fit(
    X_tr, y_tr,
    eval_set=[(X_val, y_val)],
    verbose=False
)

print(f"\nTraining complete.")
print(f"Best iteration: {xgb_model.best_iteration}")
print(f"Trees used: {xgb_model.best_iteration + 1} of {xgb_model.n_estimators} maximum")

Training (for XGB):   (1065, 446) | Failures: 71
Validation (early stop): (188, 446) | Failures: 12
Test (held-out):      (314, 446) | Failures: 21

Training complete.
Best iteration: 12
Trees used: 13 of 500 maximum


In [4]:
y_prob_train_xgb = xgb_model.predict_proba(X_tr)[:, 1]
y_prob_test_xgb  = xgb_model.predict_proba(X_test_sc)[:, 1]

pr_auc_train_xgb = average_precision_score(y_tr, y_prob_train_xgb)
pr_auc_test_xgb  = average_precision_score(y_test, y_prob_test_xgb)
roc_auc_test_xgb = roc_auc_score(y_test, y_prob_test_xgb)

print("=== Overfitting Comparison: RF vs XGBoost ===\n")
print(f"{'Model':<20} {'Train PR-AUC':>14} {'Test PR-AUC':>12} {'Gap':>10}")
print(f"{'Random Forest':<20} {'1.0000':>14} {'0.2743':>12} {'0.7257':>10}")
print(f"{'XGBoost':<20} {pr_auc_train_xgb:>14.4f} {pr_auc_test_xgb:>12.4f} "
      f"{pr_auc_train_xgb - pr_auc_test_xgb:>10.4f}")

=== Overfitting Comparison: RF vs XGBoost ===

Model                  Train PR-AUC  Test PR-AUC        Gap
Random Forest                1.0000       0.2743     0.7257
XGBoost                      0.9366       0.1530     0.7836


In [6]:
xgb_model_fixed = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    verbosity=0
)

xgb_model_fixed.fit(X_train_sc, y_train)

y_prob_train_fixed = xgb_model_fixed.predict_proba(X_train_sc)[:, 1]
y_prob_test_fixed  = xgb_model_fixed.predict_proba(X_test_sc)[:, 1]

pr_auc_train_fixed = average_precision_score(y_train, y_prob_train_fixed)
pr_auc_test_fixed  = average_precision_score(y_test, y_prob_test_fixed)
roc_auc_test_fixed = roc_auc_score(y_test, y_prob_test_fixed)

print("=== Fixed Trees XGBoost vs Random Forest ===\n")
print(f"{'Model':<25} {'Train PR-AUC':>14} {'Test PR-AUC':>12} {'Gap':>10}")
print(f"{'Random Forest':<25} {'1.0000':>14} {'0.2743':>12} {'0.7257':>10}")
print(f"{'XGBoost (early stop)':<25} {pr_auc_train_xgb:>14.4f} {pr_auc_test_xgb:>12.4f} "
      f"{pr_auc_train_xgb - pr_auc_test_xgb:>10.4f}")
print(f"{'XGBoost (200 trees)':<25} {pr_auc_train_fixed:>14.4f} {pr_auc_test_fixed:>12.4f} "
      f"{pr_auc_train_fixed - pr_auc_test_fixed:>10.4f}")

=== Fixed Trees XGBoost vs Random Forest ===

Model                       Train PR-AUC  Test PR-AUC        Gap
Random Forest                     1.0000       0.2743     0.7257
XGBoost (early stop)              0.9366       0.1530     0.7836
XGBoost (200 trees)               1.0000       0.2494     0.7506


In [7]:
thresholds = np.arange(0.05, 0.95, 0.05)
results_xgb = []

for thresh in thresholds:
    y_pred_t = (y_prob_test_xgb >= thresh).astype(int)
    tp = ((y_pred_t == 1) & (y_test == 1)).sum()
    fp = ((y_pred_t == 1) & (y_test == 0)).sum()
    fn = ((y_pred_t == 0) & (y_test == 1)).sum()

    recall_t    = tp / (tp + fn) if (tp + fn) > 0 else 0
    precision_t = tp / (tp + fp) if (tp + fp) > 0 else 0
    f1_t        = (2 * precision_t * recall_t /
                   (precision_t + recall_t)
                   if (precision_t + recall_t) > 0 else 0)

    results_xgb.append({
        'threshold': round(thresh, 2),
        'recall': recall_t,
        'precision': precision_t,
        'f1': f1_t,
        'failures_caught': int(tp),
        'false_alarms': int(fp)
    })

xgb_thresh_df = pd.DataFrame(results_xgb)
best_xgb      = xgb_thresh_df.loc[xgb_thresh_df['f1'].idxmax()]

print(f"XGBoost optimal threshold (max F1): {best_xgb['threshold']:.2f}")
print(f"  Failures caught: {best_xgb['failures_caught']} of {y_test.sum()}")
print(f"  False alarms:    {best_xgb['false_alarms']}")
print(f"  Recall:          {best_xgb['recall']:.3f}")
print(f"  Precision:       {best_xgb['precision']:.3f}")
print(f"  F1:              {best_xgb['f1']:.3f}")

XGBoost optimal threshold (max F1): 0.35
  Failures caught: 19.0 of 21
  False alarms:    183.0
  Recall:          0.905
  Precision:       0.094
  F1:              0.170


In [8]:
comparison = pd.DataFrame([
    {
        'Model': 'Naive Baseline',
        'PR-AUC': round(y_test.mean(), 4),
        'ROC-AUC': 0.5000,
        'Train PR-AUC': '-',
        'Overfitting Gap': '-',
        'Failures Caught': 0,
        'False Alarms': 0,
        'Optimal Threshold': '-'
    },
    {
        'Model': 'Logistic Regression',
        'PR-AUC': 0.1497,
        'ROC-AUC': 0.6208,
        'Train PR-AUC': '-',
        'Overfitting Gap': '-',
        'Failures Caught': 4,
        'False Alarms': 32,
        'Optimal Threshold': 0.55
    },
    {
        'Model': 'Random Forest',
        'PR-AUC': 0.2743,
        'ROC-AUC': 0.8029,
        'Train PR-AUC': 1.0000,
        'Overfitting Gap': 0.7257,
        'Failures Caught': 13,
        'False Alarms': 52,
        'Optimal Threshold': 0.20
    },
    {
        'Model': 'XGBoost (200 trees)',
        'PR-AUC': round(pr_auc_test_fixed, 4),
        'ROC-AUC': round(roc_auc_test_fixed, 4),
        'Train PR-AUC': round(pr_auc_train_fixed, 4),
        'Overfitting Gap': round(pr_auc_train_fixed - pr_auc_test_fixed, 4),
        'Failures Caught': 19,
        'False Alarms': 183,
        'Optimal Threshold': 0.35
    }
])

print(comparison.to_string(index=False))
comparison.to_csv(PROCESSED / 'model_comparison.csv', index=False)

              Model  PR-AUC  ROC-AUC Train PR-AUC Overfitting Gap  Failures Caught  False Alarms Optimal Threshold
     Naive Baseline  0.0669   0.5000            -               -                0             0                 -
Logistic Regression  0.1497   0.6208            -               -                4            32              0.55
      Random Forest  0.2743   0.8029          1.0          0.7257               13            52               0.2
XGBoost (200 trees)  0.2494   0.7382          1.0          0.7506               19           183              0.35


In [9]:
# Compare which specific failures each model caught and missed
y_prob_rf  = joblib.load(PROCESSED / 'scaler.pkl')  # placeholder
# Retrain RF quickly for direct comparison
from sklearn.ensemble import RandomForestClassifier
rf_quick = RandomForestClassifier(n_estimators=300, min_samples_leaf=2,
                                   class_weight='balanced', max_features='sqrt',
                                   random_state=42, n_jobs=-1)
rf_quick.fit(pd.DataFrame(scaler.transform(X_train), columns=X_train.columns), y_train)
rf_probs = rf_quick.predict_proba(X_test_sc)[:, 1]

# At each model's optimal threshold
rf_pred  = (rf_probs >= 0.20).astype(int)
xgb_pred = (y_prob_test_xgb >= best_xgb['threshold']).astype(int)

true_failures = y_test[y_test == 1].index.tolist()

print("=== Failure Detection Comparison: RF vs XGBoost ===\n")
print(f"{'Wafer Index':<15} {'Actual':<10} {'RF Pred':<12} {'XGB Pred':<12} {'Agreement'}")
print("-" * 60)

rf_caught   = []
xgb_caught  = []
both_caught = []
xgb_only    = []
rf_only     = []
both_missed = []

for idx in true_failures:
    pos       = y_test.index.get_loc(idx)
    rf_p      = rf_pred[pos]
    xgb_p     = xgb_pred[pos]
    agreement = "✓ Both" if rf_p == xgb_p else "✗ Differ"

    if rf_p == 1 and xgb_p == 1:
        both_caught.append(idx)
        tag = "BOTH caught"
    elif rf_p == 0 and xgb_p == 0:
        both_missed.append(idx)
        tag = "BOTH missed"
    elif rf_p == 1 and xgb_p == 0:
        rf_only.append(idx)
        tag = "RF only"
    else:
        xgb_only.append(idx)
        tag = "XGB only"

    print(f"{idx:<15} {'FAIL':<10} {'CAUGHT' if rf_p else 'MISSED':<12} "
          f"{'CAUGHT' if xgb_p else 'MISSED':<12} {tag}")

print(f"\n=== Summary ===")
print(f"Caught by both:      {len(both_caught)}")
print(f"Caught by RF only:   {len(rf_only)}")
print(f"Caught by XGB only:  {len(xgb_only)}")
print(f"Missed by both:      {len(both_missed)}")

=== Failure Detection Comparison: RF vs XGBoost ===

Wafer Index     Actual     RF Pred      XGB Pred     Agreement
------------------------------------------------------------
37              FAIL       MISSED       CAUGHT       XGB only
56              FAIL       MISSED       MISSED       BOTH missed
60              FAIL       CAUGHT       CAUGHT       BOTH caught
73              FAIL       MISSED       CAUGHT       XGB only
78              FAIL       CAUGHT       CAUGHT       BOTH caught
82              FAIL       CAUGHT       CAUGHT       BOTH caught
88              FAIL       CAUGHT       CAUGHT       BOTH caught
110             FAIL       CAUGHT       CAUGHT       BOTH caught
142             FAIL       MISSED       CAUGHT       XGB only
181             FAIL       CAUGHT       CAUGHT       BOTH caught
184             FAIL       CAUGHT       CAUGHT       BOTH caught
185             FAIL       CAUGHT       CAUGHT       BOTH caught
187             FAIL       CAUGHT       CAUGHT      